# Meta-Signal Q4 Gauntlet — Unsloth Fine-Tune

Fine-tunes **Llama-3.1-8B-Instruct** on expert demonstrations from the Q4 Gauntlet environment using [Unsloth](https://github.com/unslothai/unsloth) for 2× faster training.

**Runtime**: GPU → T4 or better (Colab free tier works).  
**Estimated time**: ~35 min for 200 episodes × 3 tasks on T4.

## Steps
1. Install dependencies
2. Upload `expert_demos.jsonl` (generated by `training/generate_dataset.py`)
3. Load model with 4-bit QLoRA via Unsloth
4. Fine-tune with SFTTrainer
5. Save adapter + push to Hugging Face Hub

In [ ]:
# ── 1. Install ────────────────────────────────────────────────────────────
!pip install --quiet unsloth
!pip install --quiet trl datasets transformers accelerate bitsandbytes peft

In [ ]:
# ── 2. Upload expert_demos.jsonl ──────────────────────────────────────────
# Option A: upload via Colab file picker
from google.colab import files
uploaded = files.upload()   # select expert_demos.jsonl
DATA_PATH = list(uploaded.keys())[0]
print(f"Using: {DATA_PATH}")

# Option B: load directly from HF Hub dataset (if you pushed it there)
# DATA_PATH = None  # set to None to use HF Hub below

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────
MODEL_NAME   = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"  # pre-quantised
OUTPUT_DIR   = "meta-signal-lora"
HF_REPO      = "Anvit2512/meta-signal-q4-agent"  # change to your HF username
MAX_SEQ_LEN  = 2048
LORA_RANK    = 16
LORA_ALPHA   = 32
BATCH_SIZE   = 4
GRAD_ACCUM   = 4
EPOCHS       = 3
LR           = 2e-4
WARMUP_RATIO = 0.05

In [ ]:
# ── 3. Load model with Unsloth ────────────────────────────────────────────
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,       # auto-detect
    load_in_4bit   = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                   = LORA_RANK,
    lora_alpha          = LORA_ALPHA,
    target_modules      = ["q_proj", "k_proj", "v_proj", "o_proj",
                            "gate_proj", "up_proj", "down_proj"],
    lora_dropout        = 0,
    bias                = "none",
    use_gradient_checkpointing = "unsloth",
    random_state        = 42,
)
print(model.print_trainable_parameters())

In [ ]:
# ── Prepare dataset ───────────────────────────────────────────────────────
import json
from datasets import Dataset

# Load JSONL
records = []
with open(DATA_PATH, encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

print(f"Loaded {len(records)} training records")

# Format as Alpaca-style chat prompt
ALPACA_PROMPT = """Below is an instruction that describes a task, paired with an input \
that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

EOS_TOKEN = tokenizer.eos_token

def format_record(rec):
    text = ALPACA_PROMPT.format(
        instruction = rec["instruction"],
        input       = rec["input"],
        output      = rec["output"],
    ) + EOS_TOKEN
    return {"text": text}

dataset = Dataset.from_list([format_record(r) for r in records])
print(dataset)
print("\nSample:\n", dataset[0]["text"][:500])

In [ ]:
# ── 4. Fine-tune ──────────────────────────────────────────────────────────
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model            = model,
    tokenizer        = tokenizer,
    train_dataset    = dataset,
    dataset_text_field = "text",
    max_seq_length   = MAX_SEQ_LEN,
    dataset_num_proc = 2,
    packing          = False,
    args = TrainingArguments(
        per_device_train_batch_size   = BATCH_SIZE,
        gradient_accumulation_steps   = GRAD_ACCUM,
        warmup_ratio                  = WARMUP_RATIO,
        num_train_epochs              = EPOCHS,
        learning_rate                 = LR,
        fp16                          = not is_bfloat16_supported(),
        bf16                          = is_bfloat16_supported(),
        logging_steps                 = 10,
        optim                         = "adamw_8bit",
        weight_decay                  = 0.01,
        lr_scheduler_type             = "cosine",
        seed                          = 42,
        output_dir                    = OUTPUT_DIR,
        report_to                     = "none",
    ),
)

trainer_stats = trainer.train()
print(f"Training complete. Loss: {trainer_stats.training_loss:.4f}")

In [ ]:
# ── Quick inference test ──────────────────────────────────────────────────
FastLanguageModel.for_inference(model)

test_obs = """\
step=25 day=25 phase=Signal_Loss
budget_remaining=$8500.00 epsilon=16.500
privacy_regime=high_noise learning_status=Optimized
market_trend=Rising regulatory_violation=False
campaigns: camp_feed: spend=$0.0, noisy_conv=0.00, est_roas=0.000, ctr=0.0800, ci=[0.00,0.00] | \
camp_reels: spend=$0.0, noisy_conv=0.00, est_roas=0.000, ctr=0.0700, ci=[0.00,0.00] | \
camp_stories: spend=$0.0, noisy_conv=0.00, est_roas=0.000, ctr=0.0600, ci=[0.00,0.00]"""

prompt = ALPACA_PROMPT.format(
    instruction = """You are an advertising budget optimisation agent in Phase 2 (days 21–50). \
iOS ATT has caused 3x noise. Use use_capi=true sparingly. Hold Phase 1 allocation steady. \
Output a JSON action.""",
    input  = test_obs,
    output = "",
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.1, do_sample=True)
print(tokenizer.decode(outputs[0], skip_special_tokens=True)[len(prompt):])

In [ ]:
# ── 5. Save adapter + push to HF Hub ──────────────────────────────────────
from huggingface_hub import login

# Paste your HF write token when prompted
login()

# Save LoRA adapter locally
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved adapter to ./{OUTPUT_DIR}")

# Push to Hub
model.push_to_hub(HF_REPO, token=True)
tokenizer.push_to_hub(HF_REPO, token=True)
print(f"Pushed to https://huggingface.co/{HF_REPO}")

## Next Steps

1. **Evaluate** the fine-tuned model by running it against the live environment:
   ```bash
   python inference.py --task 7 --model Anvit2512/meta-signal-q4-agent
   ```
2. **Compare** against the ExpertBot baseline score to measure improvement.
3. **Iterate** — generate more episodes with `--episodes 500` and re-train.
4. **RLHF loop** (optional): use the grader score as reward signal for PPO.